<a href="https://colab.research.google.com/github/byu-cs-452/byu-cs-452-class-content/blob/main/sparksql/01%20-%20SparkHelloWorld.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Setup Spark SQL
# Note if running locally you need the JVM https://www.oracle.com/java/technologies/downloads/
# Consider running in https://colab.research.google.com/
%pip install pyspark

In [ ]:
import os, sys
# Must run BEFORE the SparkSession is created (Windows: otherwise workers look for "python3")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import re
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("BibleCounter").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel('WARN')

In [ ]:
# Download the bible as txt
!curl -L "https://raw.githubusercontent.com/byu-cs-452/byu-cs-452-class-content/main/sparksql/data/bible.txt" > bible.txt

In [ ]:
# Use SparkSQL to list all the words in the bible with their counts sorted descending by count.
# Turn in your code and sample of your results (at least top 20).
# Be sure to ignore case and punctuation and eliminate the blank lines.
import re

# Read every line of the bible
lines = sc.textFile("bible.txt")

# Make everything lowercase so "The" and "the" count as the same word
lower_lines = lines.map(lambda line: line.lower())

# Split each line on anything that isn't a letter (removes punctuation and numbers)
words = lower_lines.flatMap(lambda line: re.split('[^a-z]', line))

# Remove the empty strings left over from blank lines and punctuation
real_words = words.filter(lambda word: word != '')

# Start each word with a count of 1 -> (word, 1)
word_ones = real_words.map(lambda word: (word, 1))

# Add up the counts for matching words -> (word, total)
word_counts = word_ones.reduceByKey(lambda count1, count2: count1 + count2)

# Sort by count, biggest first
sorted_word_counts = word_counts.sortBy(lambda pair: pair[1], False)

# Show the top 20
sorted_word_counts.take(20)

In [ ]:
# What to try something else?